In [15]:
from celery import Celery
import os

def build_broker_url():
    user = os.environ.get('user', 'celery')
    password = os.environ.get('password', 'celery')
    host = os.environ.get('celery_host', '192.168.1.110')
    port = os.environ.get('celery_port', '31672')
    vhost = os.environ.get('celery_vhost', 'celery')
    return f'amqp://{user}:{password}@{host}:{port}/{vhost}'

broker_url = build_broker_url()
app = Celery('nb_sender', broker=broker_url)
print('Broker URL:', broker_url)

In [16]:
from celery import Celery
import sqlite3
import uuid
from pathlib import Path
import os
from kombu import Exchange, Queue

# Configure Celery app for the database writer
def build_db_writer_broker_url():
    user = os.environ.get('user', 'celery')
    password = os.environ.get('password', 'celery')
    host = os.environ.get('celery_host', '192.168.1.110')
    port = os.environ.get('celery_port', '31672')
    vhost = os.environ.get('celery_vhost', 'celery')  # Changed from 'test' to 'celery'
    return f'amqp://{user}:{password}@{host}:{port}/{vhost}'

app = Celery('tasks', broker=build_db_writer_broker_url())

# Configure queue with priority support
app.conf.task_queues = (
    Queue(
        'database_operation',
        Exchange('tasks', type='direct'),
        routing_key='database_operation',
        queue_arguments={'x-max-priority': 10},
    ),
)

# Configure to run only 1 task at a time
app.conf.worker_concurrency = 1
app.conf.worker_prefetch_multiplier = 1
app.conf.task_default_priority = 5

app.conf.task_routes = {
    'tasks.database_operation': {'queue': 'database_operation'}
}

print('Worker configuration:')
print(f'  Broker: {build_db_writer_broker_url()}')
print('  Queue: database_operation')
print('  Concurrency: 1')
print('  Priority support: x-max-priority=10')

@app.task(name='tasks.database_operation')
def database_operation(table, operation, value):
    """
    Handle database operations for various tables
    
    Args:
        table (str): Table name (e.g., 'files', 'directories', 'encode', etc.)
        operation (str): Operation type ('write', 'update', 'delete')
        value (dict): Data to be written/updated/deleted
    
    Returns:
        dict: Result with success status and details
    """
    db_path = Path.cwd() / 'boilest.db'
    
    print(f'[database_operation] Processing: table={table}, operation={operation}')

    try:
        if table == 'files':
            return handle_files_table(db_path, operation, value)
        else:
            return {'success': False, 'error': f'Table {table} not implemented'}
    except Exception as e:
        return {'success': False, 'error': str(e)}


def handle_files_table(db_path, operation, value):
    """Handle operations for the files table"""
    conn = sqlite3.connect(str(db_path))
    cur = conn.cursor()

    try:
        if operation == 'write':
            # Insert a new file record only if file_path doesn't exist
            directory_guid = value.get('directory_guid')
            file_path = value.get('file_path')
            file_name = value.get('file_name')
            
            if not all([directory_guid, file_path, file_name]):
                return {'success': False, 'error': 'Missing required fields: directory_guid, file_path, file_name'}
            
            # Check if file_path already exists
            cur.execute('SELECT guid FROM files WHERE file_path = ?', (file_path,))
            existing = cur.fetchone()
            
            if existing:
                return {'success': False, 'error': 'File path already exists', 'existing_guid': existing[0]}
            
            # File path doesn't exist, insert new record
            guid = value.get('guid', str(uuid.uuid4()))
            cur.execute(
                'INSERT INTO files (guid, directory_guid, file_path, file_name) VALUES (?, ?, ?, ?)',
                (guid, directory_guid, file_path, file_name)
            )
            conn.commit()
            print(f'[database_operation] Inserted file: {file_path} (guid={guid})')
            return {'success': True, 'guid': guid, 'operation': 'write'}
        else:
            return {'success': False, 'error': f'Operation {operation} not implemented'}
    
    except sqlite3.IntegrityError as e:
        return {'success': False, 'error': f'Database integrity error: {str(e)}'}
    except Exception as e:
        return {'success': False, 'error': f'Database error: {str(e)}'}
    finally:
        conn.close()

print("Database writer task 'tasks.database_operation' registered and ready to listen")

In [18]:
# Start the Celery worker (this will block and listen for tasks)
print("Starting Celery worker for 'database_operation' queue...")
print("Listening for tasks... Press Ctrl+C to stop")
print("-" * 80)

app.worker_main([
    'worker',
    '--loglevel=info',
    '--concurrency=1',
    '--prefetch-multiplier=1',
    '-Q', 'database_operation',  # Only listen on this queue
])


In [ ]:
import subprocess
import json

def check_rabbitmq_queues(vhost='celery', queue_name=None):
    """
    Check RabbitMQ queues using rabbitmqctl list_queues
    
    Args:
        vhost (str): Virtual host to query (default: 'celery')
        queue_name (str): Optional filter for specific queue name
    
    Returns:
        dict: Queue information or error details
    """
    try:
        # Run rabbitmqctl list_queues command
        # Format: rabbitmqctl list_queues -p <vhost> name messages consumers
        cmd = ['rabbitmqctl', 'list_queues', '-p', vhost, 'name', 'messages', 'consumers', 'arguments']
        
        print(f"Running: {' '.join(cmd)}")
        print("-" * 80)
        
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode != 0:
            print(f"✗ Error running rabbitmqctl: {result.stderr}")
            return {'error': result.stderr}
        
        lines = result.stdout.strip().split('\n')
        queues = []
        
        # Parse the output (skip header lines)
        for line in lines:
            line = line.strip()
            if line and not line.startswith('Listing') and not line.startswith('...'):
                parts = line.split('\t')
                if len(parts) >= 3:
                    queue_info = {
                        'name': parts[0],
                        'messages': parts[1] if len(parts) > 1 else '0',
                        'consumers': parts[2] if len(parts) > 2 else '0',
                        'arguments': parts[3] if len(parts) > 3 else 'none'
                    }
                    queues.append(queue_info)
        
        # Filter by queue name if specified
        if queue_name:
            queues = [q for q in queues if q['name'] == queue_name]
            if not queues:
                print(f"✗ Queue '{queue_name}' not found in vhost '{vhost}'")
                return {'error': f"Queue '{queue_name}' not found"}
        
        # Print results
        print(f"Queues in vhost '{vhost}':")
        print("-" * 80)
        for q in queues:
            print(f"Name: {q['name']}")
            print(f"  Messages: {q['messages']}")
            print(f"  Consumers: {q['consumers']}")
            print(f"  Arguments: {q['arguments']}")
            print()
        
        return {
            'vhost': vhost,
            'total_queues': len(queues),
            'queues': queues
        }
        
    except FileNotFoundError:
        error_msg = "rabbitmqctl not found. Ensure RabbitMQ is installed and in PATH."
        print(f"✗ {error_msg}")
        return {'error': error_msg}
    except Exception as e:
        print(f"✗ Exception: {e}")
        return {'error': str(e)}

# Test: Check all queues in the 'celery' vhost
print("Checking all queues in 'celery' vhost:")
print("=" * 80)
result = check_rabbitmq_queues(vhost='celery')
print()

# Test: Check specifically for 'database_operation' queue
print("Checking for 'database_operation' queue:")
print("=" * 80)
db_queue = check_rabbitmq_queues(vhost='celery', queue_name='database_operation')

if db_queue.get('queues'):
    print("✓ database_operation queue found and active")
else:
    print("✗ database_operation queue not found - may need to create it")